# Brightness/Saturation Threshold Sweep - Issue #10

**Goal:** Find the optimal brightness and saturation thresholds for cloud detection.

**Task:** Write a loop over a grid of brightness (b) and saturation (s) values that calls detect(image, b, s), score each result with evaluate(pred, true, valid), and save the results to results/rgb_sweep.csv.

This notebook implements:
1. `detect(image, b, s)` - Cloud detection using brightness and saturation thresholds
2. `evaluate(pred, true, valid)` - Evaluation metrics (precision, recall, F1)
3. Grid sweep over threshold values
4. Results analysis and best threshold selection

## 1. Environment Setup

In [ ]:
# Install required packages
!pip install -q rasterio numpy matplotlib scikit-learn pandas

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
import rasterio
from rasterio.windows import Window
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')


## 2. Cloud Detection Function

Implement `detect(image, b, s)` that uses brightness and saturation thresholds to detect clouds.

**Rationale:**
- Clouds are typically **bright** (high brightness)
- Clouds are typically **low saturation** (near-white/gray)
- This combination helps distinguish clouds from bright surfaces like sand/snow

In [ ]:
def detect(image, b, s):
    """
    Cloud detection using brightness and saturation thresholds.
    
    Parameters:
    -----------
    image : numpy array
        RGB image with shape (height, width, 3) or (3, height, width)
    b : int or float
        Brightness threshold (0-255). Pixels with brightness > b are considered.
    s : int or float
        Saturation threshold (0-1). Pixels with saturation < s are considered.
    
    Returns:
    --------
    numpy array : Binary mask where 1 = cloud, 0 = non-cloud
    """
    # Handle different input shapes
    if image.shape[0] == 3:  # (3, height, width)
        rgb = np.transpose(image, (1, 2, 0))
    else:  # (height, width, 3)
        rgb = image.copy()
    
    # Normalize to 0-1 range
    rgb_normalized = rgb.astype(float) / 255.0
    
    # Calculate brightness (average of RGB channels)
    brightness = np.mean(rgb_normalized, axis=2)
    
    # Calculate saturation using HSV conversion
    # Saturation = (max - min) / max for non-zero max, else 0
    max_rgb = np.max(rgb_normalized, axis=2)
    min_rgb = np.min(rgb_normalized, axis=2)
    
    # Avoid division by zero
    saturation = np.zeros_like(max_rgb)
    mask = max_rgb > 0
    saturation[mask] = (max_rgb[mask] - min_rgb[mask]) / max_rgb[mask]
    
    # Normalize brightness threshold to 0-1 range
    b_normalized = b / 255.0
    
    # Apply thresholds: bright AND low-saturation pixels are clouds
    cloud_mask = (brightness > b_normalized) & (saturation < s)
    
    return cloud_mask.astype(np.uint8)


## 3. Evaluation Function

Implement `evaluate(pred, true, valid)` that calculates precision, recall, and F1 score.

In [ ]:
def evaluate(pred, true, valid):
    """
    Evaluate cloud detection predictions against ground truth.
    
    Parameters:
    -----------
    pred : numpy array
        Predicted binary mask (1 = cloud, 0 = non-cloud)
    true : numpy array
        Ground truth binary mask (1 = cloud, 0 = non-cloud)
    valid : numpy array
        Valid pixel mask (1 = valid, 0 = invalid/ignore)
    
    Returns:
    --------
    dict : Dictionary with precision, recall, and f1 scores
    """
    # Ensure arrays are flat and matching shape
    pred_flat = pred.flatten().astype(int)
    true_flat = true.flatten().astype(int)
    valid_flat = valid.flatten().astype(int)
    
    # Only consider valid pixels
    valid_indices = valid_flat == 1
    
    if np.sum(valid_indices) == 0:
        return {
            'precision': 0.0,
            'recall': 0.0,
            'f1': 0.0
        }
    
    pred_valid = pred_flat[valid_indices]
    true_valid = true_flat[valid_indices]
    
    # Handle edge cases where all predictions are the same
    if len(np.unique(pred_valid)) == 1 or len(np.unique(true_valid)) == 1:
        # Calculate metrics manually
    
    try:
        precision = precision_score(true_valid, pred_valid, zero_division=0)
        recall = recall_score(true_valid, pred_valid, zero_division=0)
        
        # Handle F1 when both precision and recall are 0
        if precision == 0 and recall == 0:
            f1 = 0.0
        else:
            f1 = f1_score(true_valid, pred_valid, zero_division=0)
    except:
        # Fallback to manual calculation
        tp = np.sum((pred_valid == 1) & (true_valid == 1))
        fp = np.sum((pred_valid == 1) & (true_valid == 0))
        fn = np.sum((pred_valid == 0) & (true_valid == 1))
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1)
    }


## 4. Test Functions with Sample Data

Create synthetic test data to verify the functions work correctly.

In [ ]:
# Create synthetic test image
test_image = np.random.randint(0, 256, (100, 100, 3), dtype=np.uint8)

# Create some "cloud" pixels (bright, low saturation)
test_image[20:40, 20:40] = [240, 240, 240]  # Bright white (cloud-like)
test_image[60:80, 60:80] = [30, 30, 30]     # Dark (non-cloud)
test_image[30:50, 70:90] = [200, 50, 50]    # Bright red (high saturation, non-cloud)

# Test detect function
cloud_mask = detect(test_image, b=200, s=0.3)

print(f"Test image shape: {test_image.shape}")
print(f"Cloud mask shape: {cloud_mask.shape}")
print(f"Cloud pixels detected: {np.sum(cloud_mask)}")
print(f"Cloud coverage: {np.mean(cloud_mask) * 100:.2f}%")

# Create synthetic ground truth and valid mask
true_mask = np.zeros((100, 100), dtype=np.uint8)
true_mask[20:40, 20:40] = 1  # True cloud region

valid_mask = np.ones((100, 100), dtype=np.uint8)  # All pixels valid

# Test evaluate function
metrics = evaluate(cloud_mask, true_mask, valid_mask)
print(f"\nEvaluation metrics:")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall: {metrics['recall']:.4f}")
print(f"F1 Score: {metrics['f1']:.4f}")

## 5. Grid Sweep Implementation

Implement the grid sweep over brightness and saturation values.

In [ ]:
def brightness_saturation_sweep(image, true_mask, valid_mask, 
                                 brightness_range, saturation_range):
    """
    Perform grid sweep over brightness and saturation thresholds.
    
    Parameters:
    -----------
    image : numpy array
        RGB image
    true_mask : numpy array
        Ground truth binary mask
    valid_mask : numpy array
        Valid pixel mask
    brightness_range : list or array
        List of brightness threshold values to test
    saturation_range : list or array
        List of saturation threshold values to test
    
    Returns:
    --------
    pandas.DataFrame : Results with columns: b, s, precision, recall, f1
    """
    results = []
    total_combinations = len(brightness_range) * len(saturation_range)
    
    print(f"Starting grid sweep: {total_combinations} threshold combinations")
    
    for i, b in enumerate(brightness_range):
        for j, s in enumerate(saturation_range):
            # Detect clouds with current thresholds
            pred_mask = detect(image, b, s)
            
            # Evaluate predictions
            metrics = evaluate(pred_mask, true_mask, valid_mask)
            
            # Store results
            results.append({
                'b': b,
                's': s,
                'precision': metrics['precision'],
                'recall': metrics['recall'],
                'f1': metrics['f1']
            })
            
            # Progress update
            if (i * len(saturation_range) + j + 1) % 10 == 0:
                progress = (i * len(saturation_range) + j + 1) / total_combinations * 100
                print(f"Progress: {progress:.1f}% ({i * len(saturation_range) + j + 1}/{total_combinations})")
    
    results_df = pd.DataFrame(results)
    print(f"Grid sweep completed! Total results: {len(results_df)}")
    
    return results_df


## 6. Run Sweep on Test Data

Test the sweep implementation with our synthetic data.

In [ ]:
# Define threshold ranges for testing
brightness_values = [150, 175, 200, 225, 250]
saturation_values = [0.1, 0.2, 0.3, 0.4, 0.5]

# Run sweep on test data
test_results = brightness_saturation_sweep(
    test_image, true_mask, valid_mask, 
    brightness_values, saturation_values
)

print("\nFirst few results:")
print(test_results.head())

## 7. Find Best Thresholds

Identify the best brightness and saturation thresholds based on F1 score.

In [ ]:
def find_best_thresholds(results_df, metric='f1'):
    """
    Find the best threshold combination based on specified metric.
    
    Parameters:
    -----------
    results_df : pandas.DataFrame
        Results from grid sweep
    metric : str
        Metric to optimize ('f1', 'precision', or 'recall')
    
    Returns:
    --------
    dict : Best threshold combination and metrics
    """
    best_idx = results_df[metric].idxmax()
    best_row = results_df.loc[best_idx]
    
    return {
        'brightness': int(best_row['b']),
        'saturation': float(best_row['s']),
        'precision': float(best_row['precision']),
        'recall': float(best_row['recall']),
        'f1': float(best_row['f1'])
    }

# Find best thresholds by F1
best_thresholds = find_best_thresholds(test_results, metric='f1')

print("Best thresholds by F1 score:")
print(f"Brightness: {best_thresholds['brightness']}")
print(f"Saturation: {best_thresholds['saturation']}")
print(f"Precision: {best_thresholds['precision']:.4f}")
print(f"Recall: {best_thresholds['recall']:.4f}")
print(f"F1 Score: {best_thresholds['f1']:.4f}")

## 8. Save Results to CSV

Save the sweep results to results/rgb_sweep.csv as specified in the task.

In [ ]:
# Ensure results directory exists
results_dir = Path('../results')
results_dir.mkdir(exist_ok=True)

# Save results to CSV
output_path = results_dir / 'rgb_sweep.csv'
test_results.to_csv(output_path, index=False)

print(f"Results saved to: {output_path}")
print(f"Total results: {len(test_results)}")

## 9. Visualize Results

Create visualizations to understand the threshold performance.

In [ ]:
# Create pivot table for heatmap visualization
pivot_f1 = test_results.pivot(index='s', columns='b', values='f1')
pivot_precision = test_results.pivot(index='s', columns='b', values='precision')
pivot_recall = test_results.pivot(index='s', columns='b', values='recall')

# Plot heatmaps
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# F1 Score heatmap
im1 = axes[0].imshow(pivot_f1, cmap='viridis', aspect='auto')
axes[0].set_title('F1 Score')
axes[0].set_xlabel('Brightness Threshold')
axes[0].set_ylabel('Saturation Threshold')
axes[0].set_xticks(range(len(brightness_values)))
axes[0].set_xticklabels(brightness_values)
axes[0].set_yticks(range(len(saturation_values)))
axes[0].set_yticklabels(saturation_values)
plt.colorbar(im1, ax=axes[0])

# Precision heatmap
im2 = axes[1].imshow(pivot_precision, cmap='viridis', aspect='auto')
axes[1].set_title('Precision')
axes[1].set_xlabel('Brightness Threshold')
axes[1].set_ylabel('Saturation Threshold')
axes[1].set_xticks(range(len(brightness_values)))
axes[1].set_xticklabels(brightness_values)
axes[1].set_yticks(range(len(saturation_values)))
axes[1].set_yticklabels(saturation_values)
plt.colorbar(im2, ax=axes[1])

# Recall heatmap
im3 = axes[2].imshow(pivot_recall, cmap='viridis', aspect='auto')
axes[2].set_title('Recall')
axes[2].set_xlabel('Brightness Threshold')
axes[2].set_ylabel('Saturation Threshold')
axes[2].set_xticks(range(len(brightness_values)))
axes[2].set_xticklabels(brightness_values)
axes[2].set_yticks(range(len(saturation_values)))
axes[2].set_yticklabels(saturation_values)
plt.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.show()


## 10. Integration with Real Data

TODO: When real VHR imagery and ground truth masks are available, replace the synthetic test data with real data.

**Required inputs:**
- Real VHR RGB imagery (from Maxar Open Data or other sources)
- Ground truth cloud masks (hand-labeled or from reference dataset)
- Valid pixel masks (to exclude edges, invalid regions, etc.)

In [ ]:
# Example of how to use with real data (when available):
'''
# Load real image
with rasterio.open('path/to/real_image.tif') as src:
    real_image = src.read()
    # Transpose to (height, width, bands) if needed
    if real_image.shape[0] == 3:
        real_image = np.transpose(real_image, (1, 2, 0))

# Load ground truth mask
real_true_mask = ...  # Load from file

# Create valid mask (all pixels valid, or exclude edges)
real_valid_mask = np.ones(real_image.shape[:2], dtype=np.uint8)

# Define comprehensive threshold ranges
brightness_range = np.arange(100, 256, 10)  # 100 to 255, step 10
saturation_range = np.arange(0.05, 0.8, 0.05)  # 0.05 to 0.75, step 0.05

# Run comprehensive sweep
real_results = brightness_saturation_sweep(
    real_image, real_true_mask, real_valid_mask,
    brightness_range, saturation_range
)

# Save comprehensive results
real_results.to_csv('../results/rgb_sweep_comprehensive.csv', index=False)

# Find and report best thresholds
best_real = find_best_thresholds(real_results, metric='f1')
print(f"Best thresholds for real data: b={best_real['brightness']}, s={best_real['saturation']}")
print(f"Metrics: Precision={best_real['precision']:.4f}, Recall={best_real['recall']:.4f}, F1={best_real['f1']:.4f}")
'''

print("Real data integration template ready (commented out until data is available)")

## 11. Summary and Next Steps

**Completed:**
- ✅ Created `detect(image, b, s)` function for brightness/saturation thresholding
- ✅ Created `evaluate(pred, true, valid)` function for precision/recall/F1 metrics
- ✅ Implemented grid sweep over threshold values
- ✅ Saved results to `results/rgb_sweep.csv`
- ✅ Found best thresholds by F1 score

**Next Steps:**
- Replace synthetic test data with real VHR imagery
- Create or obtain ground truth cloud masks
- Run comprehensive sweep with wider threshold ranges
- Analyze results and document findings
- Compare with other classical methods (dark-channel prior, HOT index)

**File Structure:**
- `notebooks/02_brightness_saturation_sweep.ipynb` - This notebook
- `results/rgb_sweep.csv` - Sweep results (will be created when run with real data)